In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import coo_matrix, csr_matrix, hstack, bmat, diags
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr
from sklearn.preprocessing import StandardScaler

# ============================================================
# Load data
# ============================================================

no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
S, TT = y.shape
period = 52

# ============================================================
# logistic regression dataset (same as p01)
# ============================================================
loc = np.where(y[:, :-1] == 0)
pairs = np.column_stack(loc)
order = np.lexsort((pairs[:,0], pairs[:,1]))
pairs_R = pairs[order]
pairs_R[:,1] += 1

row_idx = pairs_R[:,0]
time_idx = pairs_R[:,1] - 1
N = len(row_idx)

next_y = y[pairs_R[:,0], pairs_R[:,1]]
kappa = next_y - 0.5

t = time_idx + 1

# ============================================================
# Covariates: ONLY intercept, cos, sin, t (IID only)
# ============================================================

covariates = np.column_stack([
    np.ones(N),                          # intercept
    np.cos(2*np.pi*t/period),            # cos
    np.sin(2*np.pi*t/period),            # sin
    t                                    # trend
])

K = covariates.shape[1]   # K = 4
print("Covariates shape:", covariates.shape)

# ============================================================
# Construct design matrix: each block is IID S×S diagonal
# ============================================================

rows, cols, vals = [], [], []
for i in tqdm(range(N), desc="Building design"):
    loc_i = row_idx[i]
    cov = covariates[i]
    for k in range(K):
        rows.append(i)
        cols.append(loc_i + k*S)
        vals.append(cov[k])

design_mat = coo_matrix((vals, (rows, cols)), shape=(N, K*S)).tocsr()

theta_dim = K * S   # no extra lat/elev
print("theta_dim =", theta_dim)

# ============================================================
# MCMC settings
# ============================================================

burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
all_tau   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

a_tau = 0.001
b_tau = 0.001

save_idx = 0

# ============================================================
# MCMC LOOP
# ============================================================

for it in tqdm(range(total_iters), desc="MCMC"):

    # 1. PG augmentation
    phi = design_mat.dot(curr_theta)
    omega = random_polyagamma(1, phi, size=N)

    # 2. Posterior precision: all IID blocks
    block_list = []
    for j in range(K):
        block_list.append((1/curr_tau[j]) * diags(np.ones(S)))  # IID only

    blocks = []
    for i in range(K):
        row = []
        for j in range(K):
            if i == j:
                row.append(block_list[i])
            else:
                row.append(None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    # 3. posterior precision
    xtOmega = design_mat.T.multiply(omega)
    pos_prec = xtOmega.dot(design_mat) + curr_prec
    factor = cholesky(pos_prec)

    # 4. sample theta
    mu = factor.solve_A(design_mat.T.dot(kappa))
    eps = np.random.randn(theta_dim)
    curr_theta = mu + factor.solve_A(eps)

    # 5. Update tau_j
    for j in range(K):
        sl = slice(j*S, (j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau + S/2, 1/(b_tau + quad/2))

    # 6. save samples
    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:, save_idx] = curr_theta
        all_tau[:, save_idx] = curr_tau
        save_idx += 1
        if save_idx >= tot_save:
            break

# ============================================================
# Save outputs
# ============================================================

np.savez_compressed(r"D:\77\Research\temp\snow\ind01.npz",
                    all_theta=all_theta,
                    all_tau=all_tau)


Covariates shape: (2800090, 4)


Building design: 100%|██████████| 2800090/2800090 [00:06<00:00, 440221.59it/s]


theta_dim = 6472


MCMC:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_59912\853640463.py:127: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(pos_prec)
MCMC: 100%|█████████▉| 5995/6000 [2:02:51<00:06,  1.23s/it]  


: 

In [7]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.sparse import coo_matrix, csr_matrix, hstack, diags, bmat
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr
from sklearn.preprocessing import StandardScaler

# ============================================================
# Load data
# ============================================================

no_nbs = np.array([57,170,236,269,343,685,946,947,989,1037,1084,1090,1109,1118,1127,1176,1203]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

df1 = snow_cleaned_full.drop(index=no_nbs)
df2 = snow_cleaned_full.iloc[no_nbs]
all_y = pd.concat([df1, df2], axis=0).reset_index(drop=True)

y = all_y.iloc[:, 2:].to_numpy()
S, TT = y.shape
period = 52

# ============================================================
# Build dataset for p10: y_t = 1 → y_{t+1} = 0
# ============================================================

loc = np.where(y[:, :-1] == 1)     # previous state = 1
pairs = np.column_stack(loc)
order = np.lexsort((pairs[:,0], pairs[:,1]))
pairs_R = pairs[order]
pairs_R[:,1] += 1                  # next time index

row_idx = pairs_R[:,0]
time_idx = pairs_R[:,1] - 1
N = len(row_idx)

next_y = y[pairs_R[:,0], pairs_R[:,1]]

# p10 event: snow disappears
event = 1 - next_y          # y_t=1 → y_{t+1}=0  → event=1
kappa = event - 0.5

t = time_idx + 1

# ============================================================
# Covariates: intercept, cos, sin, t  (IID blocks)
# ============================================================

covariates = np.column_stack([
    np.ones(N),                          # intercept
    np.cos(2*np.pi*t/period),            # cos
    np.sin(2*np.pi*t/period),            # sin
    t                                    # trend
])

K = covariates.shape[1]   # K = 4
print("Covariates shape:", covariates.shape)

# ============================================================
# Construct design matrix: IID-only model
# ============================================================

rows, cols, vals = [], [], []

for i in tqdm(range(N), desc="Building design"):
    loc_i = row_idx[i]
    cov = covariates[i]
    for k in range(K):
        rows.append(i)
        cols.append(loc_i + k*S)
        vals.append(cov[k])

design_mat = coo_matrix((vals, (rows, cols)), shape=(N, K*S)).tocsr()

theta_dim = K * S
print("theta_dim =", theta_dim)

# ============================================================
# MCMC settings
# ============================================================

burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

all_theta = np.zeros((theta_dim, tot_save))
all_tau   = np.zeros((K, tot_save))

curr_theta = np.zeros(theta_dim)
curr_tau   = np.ones(K)

a_tau = 0.001
b_tau = 0.001

save_idx = 0

# ============================================================
# MCMC LOOP
# ============================================================

for it in tqdm(range(total_iters), desc="MCMC"):

    # 1. PG augmentation
    phi = design_mat.dot(curr_theta)
    omega = random_polyagamma(1, phi, size=N)

    # 2. IID prior blocks
    block_list = []
    for j in range(K):
        block_list.append((1/curr_tau[j]) * diags(np.ones(S)))

    blocks = []
    for i in range(K):
        row = []
        for j in range(K):
            row.append(block_list[i] if i == j else None)
        blocks.append(row)

    curr_prec = bmat(blocks, format="csr")

    # 3. posterior precision
    xtOmega = design_mat.T.multiply(omega)
    pos_prec = xtOmega.dot(design_mat) + curr_prec
    factor = cholesky(pos_prec)

    # 4. sample theta
    mu = factor.solve_A(design_mat.T.dot(kappa))
    eps = np.random.randn(theta_dim)
    curr_theta = mu + factor.solve_A(eps)

    # 5. update tau
    for j in range(K):
        sl = slice(j*S, (j+1)*S)
        beta = curr_theta[sl]
        quad = beta @ beta
        curr_tau[j] = 1 / np.random.gamma(a_tau + S/2, 1/(b_tau + quad/2))

    # 6. save
    if it >= burn and ((it - burn) % thin == 0):
        all_theta[:, save_idx] = curr_theta
        all_tau[:, save_idx]   = curr_tau
        save_idx += 1

        if save_idx >= tot_save:
            break

# ============================================================
# Save outputs
# ============================================================

np.savez_compressed(r"D:\77\Research\temp\snow\ind10.npz",
                    all_theta=all_theta,
                    all_tau=all_tau)

Covariates shape: (1573364, 4)


Building design: 100%|██████████| 1573364/1573364 [00:02<00:00, 538145.28it/s]


theta_dim = 6472


MCMC:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_59912\3437129024.py:129: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(pos_prec)
MCMC: 100%|█████████▉| 5995/6000 [1:06:23<00:03,  1.51it/s]
